# Consistency in L features

In [1]:
import os
import re
import glob
from typing import List, Optional, Tuple, Dict, Literal

import numpy as np
import pandas as pd
from scipy import stats

# -------------------------
# Generic directory loader (summary_sc_<id>.csv OR turns_<id>.csv)
# -------------------------
def load_participant_matrix_from_dir(
    data_dir: str,
    *,
    pattern: str,
    id_regex: str,
    features: Optional[List[str]] = None,
    # for one-row files use "first"; for turns_<id>.csv use "median" (recommended)
    row_agg: Literal["first", "mean", "median"] = "first",
    # participant-only turns filter (best-effort heuristic)
    participant_only: bool = True,
    speaker_col_candidates: Tuple[str, ...] = (
        "speaker", "Speaker", "role", "Role", "speaker_role", "SpeakerRole", "who", "Who"
    ),
    # values that indicate interviewer/Ellie to exclude
    exclude_speakers_substrings: Tuple[str, ...] = ("ellie", "interviewer", "assistant"),
) -> pd.DataFrame:
    """
    Reads many CSVs like summary_sc_<id>.csv or turns_<id>.csv into one DataFrame:
      index = Participant (id from filename)
      columns = aggregated features

    For turns_<id>.csv (multiple rows), set row_agg="median" (default recommendation).
    """
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Not a directory: {data_dir}")

    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files found: {os.path.join(data_dir, pattern)}")

    rx = re.compile(id_regex)
    rows = []
    for fp in files:
        m = rx.search(os.path.basename(fp))
        if not m:
            continue
        pid = int(m.group(1))

        df = pd.read_csv(fp)
        if df.empty:
            continue

        # optional: keep participant-only turns by removing Ellie/interviewer rows
        if participant_only:
            spk_col = next((c for c in speaker_col_candidates if c in df.columns), None)
            if spk_col is not None:
                spk = df[spk_col].astype(str).str.lower()
                mask = np.ones(len(df), dtype=bool)
                for bad in exclude_speakers_substrings:
                    mask &= ~spk.str.contains(bad, na=False)
                df = df.loc[mask]
                if df.empty:
                    continue

        # keep only requested features (+ any needed for filtering already done)
        if features is not None:
            keep = [c for c in features if c in df.columns]
            df = df[keep].copy()

        # convert columns to numeric where possible
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        if df.empty:
            continue

        if row_agg == "first":
            rec = df.iloc[0].to_dict()
        elif row_agg == "mean":
            rec = df.mean(axis=0, numeric_only=True).to_dict()
        elif row_agg == "median":
            rec = df.median(axis=0, numeric_only=True).to_dict()
        else:
            raise ValueError("row_agg must be one of: 'first','mean','median'")

        rec["Participant"] = pid
        rows.append(rec)

    if not rows:
        raise ValueError(f"Parsed 0 valid files in: {data_dir}")

    out = pd.DataFrame(rows).set_index("Participant").sort_index()
    return out


# -------------------------
# Agreement metrics (EN vs UA)
# -------------------------
def _paired_clean(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    return a[m], b[m]


def _bootstrap_ci(values_fn, a: np.ndarray, b: np.ndarray, n_boot: int = 2000, seed: int = 1706) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    a, b = _paired_clean(a, b)
    n = len(a)
    if n < 3:
        return np.nan, np.nan, np.nan

    stats_list = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        s = values_fn(a[idx], b[idx])
        if np.isfinite(s):
            stats_list.append(s)

    if not stats_list:
        return np.nan, np.nan, np.nan

    lo, hi = np.percentile(stats_list, [2.5, 97.5])
    return float(np.mean(stats_list)), float(lo), float(hi)


def _pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3 or np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3:
        return np.nan
    r, _ = stats.spearmanr(a, b, nan_policy="omit")
    return float(r)


def icc2_1(a: np.ndarray, b: np.ndarray) -> float:
    """
    ICC(2,1): two-way random effects, absolute agreement, single measurement.
    EN and UA are the two "raters"; subjects are participants.
    """
    a, b = _paired_clean(a, b)
    n = len(a)
    k = 2
    if n < 3:
        return np.nan

    Y = np.vstack([a, b]).T  # (n, 2)
    mean_subject = Y.mean(axis=1, keepdims=True)
    mean_rater = Y.mean(axis=0, keepdims=True)
    grand_mean = Y.mean()

    SSR = k * np.sum((mean_subject - grand_mean) ** 2)
    SSC = n * np.sum((mean_rater - grand_mean) ** 2)
    SSE = np.sum((Y - mean_subject - mean_rater + grand_mean) ** 2)

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    denom = MSR + (k - 1) * MSE + (k * (MSC - MSE) / n)
    if denom <= 0:
        return np.nan
    return float((MSR - MSE) / denom)


def _mean_shift(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) == 0:
        return np.nan
    return float(np.mean(b - a))  # UA - EN


def _median_min_max(x: np.ndarray) -> Tuple[float, float, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    return float(np.median(x)), float(np.min(x)), float(np.max(x))


# -------------------------
# Main API (works for summary_sc_*.csv AND turns_*.csv)
# -------------------------
def check_statistic(
    *,
    features: List[str],
    dataset_eng_dir: str,
    dataset_ukr_dir: str,
    file_kind: Literal["summary", "turns"] = "summary",
    n_boot: int = 2000,
    seed: int = 1706,
    strict: bool = False,
    # turns only: how to aggregate multiple turn rows -> one value per participant per feature
    turns_agg: Literal["mean", "median"] = "median",
    # turns only: try to exclude interviewer/Ellie rows
    participant_only_turns: bool = True,
) -> pd.DataFrame:
    """
    Computes EN vs UA agreement per feature:
      Pearson r, Spearman rho, ICC(2,1), mean shift Δμ = mean(UA-EN), with bootstrap 95% CI.

    If file_kind="turns", we first aggregate each turns_<id>.csv into a single row per participant
    using turns_agg (default median). This matches your “turn-level -> interview-level” aggregation logic.
    """
    if file_kind == "summary":
        pattern = "summary_sc_*.csv"
        id_regex = r"summary_sc_(\d+)\.csv$"
        row_agg = "first"
        participant_only = False  # summary is already participant-only
    elif file_kind == "turns":
        pattern = "turns_*.csv"
        id_regex = r"turns_(\d+)\.csv$"
        row_agg = turns_agg
        participant_only = participant_only_turns
    else:
        raise ValueError("file_kind must be 'summary' or 'turns'")

    df_en = load_participant_matrix_from_dir(
        dataset_eng_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )
    df_ua = load_participant_matrix_from_dir(
        dataset_ukr_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )

    merged = df_en.add_suffix("_EN").join(df_ua.add_suffix("_UA"), how="inner")
    if merged.empty:
        raise ValueError("No overlapping participants between EN and UA dirs.")

    rows: List[Dict[str, object]] = []

    for feat in features:
        c_en = f"{feat}_EN"
        c_ua = f"{feat}_UA"

        if c_en not in merged.columns or c_ua not in merged.columns:
            if strict:
                raise KeyError(f"Missing feature in merged data: {feat}")
            rows.append({
                "feature": feat,
                "n": 0,
                "EN_median[min,max]": "",
                "UA_median[min,max]": "",
                "r [95% CI]": "",
                "rho [95% CI]": "",
                "ICC(2,1) [95% CI]": "",
                "mean shift Δμ (UA-EN) [95% CI]": "",
                "note": "missing in EN/UA files",
            })
            continue

        a = merged[c_en].to_numpy(dtype=float)  # EN
        b = merged[c_ua].to_numpy(dtype=float)  # UA
        a, b = _paired_clean(a, b)
        n = len(a)

        en_med, en_min, en_max = _median_min_max(a)
        ua_med, ua_min, ua_max = _median_min_max(b)

        r_m, r_lo, r_hi = _bootstrap_ci(_pearson, a, b, n_boot=n_boot, seed=seed)
        s_m, s_lo, s_hi = _bootstrap_ci(_spearman, a, b, n_boot=n_boot, seed=seed)

        icc_mean, icc_lo, icc_hi = _bootstrap_ci(lambda x, y: icc2_1(x, y), a, b, n_boot=n_boot, seed=seed)
        sh_m, sh_lo, sh_hi = _bootstrap_ci(_mean_shift, a, b, n_boot=n_boot, seed=seed)

        rows.append({
            "feature": feat,
            "n": int(n),
            "EN_median[min,max]": f"{en_med:.3f} [{en_min:.3f},{en_max:.3f}]",
            "UA_median[min,max]": f"{ua_med:.3f} [{ua_min:.3f},{ua_max:.3f}]",
            "r [95% CI]": f"{r_m:.3f} [{r_lo:.3f},{r_hi:.3f}]",
            "rho [95% CI]": f"{s_m:.3f} [{s_lo:.3f},{s_hi:.3f}]",
            "ICC(2,1) [95% CI]": f"{icc_mean:.3f} [{icc_lo:.3f},{icc_hi:.3f}]",
            "mean shift Δμ (UA-EN) [95% CI]": f"{sh_m:.3f} [{sh_lo:.3f},{sh_hi:.3f}]",
            "note": "",
        })

    out = pd.DataFrame(rows).sort_values(["n", "feature"], ascending=[False, True]).reset_index(drop=True)
    return out


In [3]:
# ---- B0: language-independent ----
B0_SUMMARY = [
    "file_length",
    "speech_length_minutes",
    "speech_length_words",
    "words_per_min",
    "speech_percentage",
    "mean_pre_word_pause",
    "mean_pause_variability",
    "word_repeat_percentage",
    "phrase_repeat_percentage",
    # optional session/turn aggregates (still language-independent)
    "num_turns",
    "num_one_word_turns",
    "mean_turn_length_minutes",
    "mean_turn_length_words",
    "mean_pre_turn_pause",
    "speaker_percentage",
    "num_interrupts",
]

stats_df = check_statistic(
    features=B0_SUMMARY,
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_eng_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_ukr_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

/var/folders/zz/9j6cqjd91k7190vj479mr1jw0000gn/T/ipykernel_65048/2932081485.py:140: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.spearmanr(a, b, nan_policy="omit")


,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,file_length,"14.914 [6.785,32.395]","14.914 [6.785,32.395]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
1,mean_pause_variability,"0.801 [0.115,24.897]","0.613 [0.034,27.155]","0.989 [0.979,0.995]","0.970 [0.958,0.980]","0.986 [0.977,0.992]","-0.101 [-0.150,-0.054]"
2,mean_pre_turn_pause,"1.077 [0.238,4.545]","1.077 [0.238,4.545]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
3,mean_pre_word_pause,"0.265 [0.109,1.466]","0.130 [0.025,1.240]","0.953 [0.933,0.967]","0.917 [0.891,0.938]","0.694 [0.593,0.775]","-0.137 [-0.144,-0.131]"
4,mean_turn_length_minutes,"0.303 [0.038,3.565]","0.303 [0.038,3.565]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
5,mean_turn_length_words,"34.000 [3.745,340.444]","30.079 [3.447,298.222]","0.999 [0.999,1.000]","0.999 [0.999,0.999]","0.983 [0.980,0.986]","-5.320 [-5.990,-4.686]"
6,num_interrupts,"0.000 [0.000,1.000]","0.000 [0.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
7,num_one_word_turns,"3.000 [0.000,16.000]","3.000 [0.000,16.000]","0.952 [0.934,0.966]","0.945 [0.925,0.962]","0.940 [0.918,0.957]","0.344 [0.244,0.451]"
8,num_turns,"33.000 [4.000,67.000]","33.000 [4.000,67.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
9,phrase_repeat_percentage,"0.087 [0.000,6.815]","0.119 [0.000,6.568]","0.970 [0.939,0.989]","0.932 [0.891,0.966]","0.968 [0.933,0.989]","0.040 [0.019,0.064]"


In [4]:
L_SUMMARY_FEATURES = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    'sentiment_vader_pos', 'sentiment_vader_neg', 'sentiment_vader_neu',
       'sentiment_vader_overall',
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    "first_person_sentiment_overall",
       'first_person_sentiment_positive_vader',
       'first_person_sentiment_negative_vader',
       'first_person_sentiment_overall_vader',
    "word_coherence_mean",
    "word_coherence_var",
    "word_coherence_5_mean",
    "word_coherence_5_var",
    "word_coherence_10_mean",
    "word_coherence_10_var",
    *[f"word_coherence_variability_{k}_mean" for k in range(2, 11)],
    *[f"word_coherence_variability_{k}_var" for k in range(2, 11)],
    "first_order_sentence_tangeniality_mean",
    "first_order_sentence_tangeniality_var",
    "second_order_sentence_tangeniality_mean",
    "second_order_sentence_tangeniality_var",
    "turn_to_turn_tangeniality_mean",
    "turn_to_turn_tangeniality_var",
    "turn_to_turn_tangeniality_slope",
    "semantic_perplexity_mean",
    "semantic_perplexity_var",
    "semantic_perplexity_5_mean",
    "semantic_perplexity_5_var",
    "semantic_perplexity_11_mean",
    "semantic_perplexity_11_var",
    "semantic_perplexity_15_mean",
    "semantic_perplexity_15_var",
]


stats_df = check_statistic(
    features=L_SUMMARY_FEATURES,
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_eng_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#result_oppenwillis_eng_norm_gemma_updated_sentiment_FIXED_21-2-26",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_ukr_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#result_oppenwillis_ukr_norm_gemma_updated_sentiment_FIXED_21-2-26",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_order_sentence_tangeniality_mean,"0.482 [0.390,0.697]","0.467 [0.372,0.667]","0.875 [0.833,0.912]","0.847 [0.808,0.881]","0.783 [0.725,0.838]","-0.017 [-0.019,-0.015]"
1,first_order_sentence_tangeniality_var,"0.010 [0.001,0.044]","0.009 [0.000,0.048]","0.857 [0.803,0.898]","0.787 [0.726,0.840]","0.844 [0.786,0.890]","-0.001 [-0.002,-0.001]"
2,first_person_percentage,"8.767 [3.852,12.865]","6.875 [3.358,10.672]","0.870 [0.839,0.898]","0.865 [0.830,0.896]","0.465 [0.414,0.515]","-1.870 [-1.963,-1.779]"
3,first_person_sentiment_negative,"2.410 [0.714,5.865]","1.577 [0.369,4.661]","0.832 [0.786,0.873]","0.810 [0.759,0.852]","0.499 [0.431,0.566]","-0.811 [-0.867,-0.756]"
4,first_person_sentiment_negative_vader,"0.363 [0.025,1.760]","0.305 [0.011,1.251]","0.769 [0.681,0.839]","0.773 [0.707,0.829]","0.742 [0.657,0.811]","-0.056 [-0.076,-0.038]"
5,first_person_sentiment_overall,"24.258 [6.830,43.069]","25.243 [8.451,47.121]","0.845 [0.804,0.879]","0.835 [0.787,0.875]","0.835 [0.791,0.872]","0.945 [0.518,1.371]"
6,first_person_sentiment_overall_vader,"19.480 [6.688,52.964]","20.729 [5.286,61.617]","0.894 [0.855,0.926]","0.855 [0.814,0.890]","0.864 [0.821,0.900]","1.419 [1.109,1.742]"
7,first_person_sentiment_positive,"27.724 [12.590,45.251]","29.957 [14.480,48.471]","0.878 [0.844,0.907]","0.863 [0.824,0.899]","0.821 [0.776,0.858]","2.202 [1.870,2.549]"
8,first_person_sentiment_positive_vader,"20.037 [7.247,52.964]","21.379 [5.286,61.617]","0.892 [0.852,0.925]","0.850 [0.807,0.887]","0.853 [0.805,0.890]","1.656 [1.356,1.985]"
9,mattr_10,"0.919 [0.843,0.951]","0.935 [0.846,0.975]","0.848 [0.806,0.883]","0.829 [0.780,0.870]","0.573 [0.504,0.640]","0.015 [0.014,0.016]"


In [5]:
L_TURNS = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    # discourse (per-turn)
    "first_order_sentence_tangeniality",
    "second_order_sentence_tangeniality",
    "turn_to_turn_tangeniality",
    "semantic_perplexity",
    "semantic_perplexity_5",
    "semantic_perplexity_11",
    "semantic_perplexity_15",
]

# 2) Turns files (turns_<pid>.csv), aggregated per participant by median:
stats_turns = check_statistic(
    features=L_TURNS,  # your per-turn feature list
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_eng_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#result_oppenwillis_eng_norm_gemma_updated_sentiment_FIXED_21-2-26",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/result_phonoma_gemma_ukr_3-04-2026",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#result_oppenwillis_ukr_norm_gemma_updated_sentiment_FIXED_21-2-26",
    file_kind="turns",
    turns_agg="mean",
    # participant_only=True,
)

stats_turns.drop(columns=["note", 'n'], inplace=True)
stats_turns

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_order_sentence_tangeniality,"0.482 [0.390,0.697]","0.467 [0.372,0.667]","0.875 [0.833,0.912]","0.847 [0.808,0.881]","0.783 [0.725,0.838]","-0.017 [-0.019,-0.015]"
1,first_person_percentage,"7.854 [2.526,13.725]","6.086 [1.637,13.047]","0.875 [0.843,0.902]","0.847 [0.799,0.887]","0.552 [0.491,0.612]","-1.774 [-1.876,-1.676]"
2,first_person_sentiment_negative,"2.410 [0.714,5.865]","1.577 [0.369,4.661]","0.832 [0.786,0.873]","0.810 [0.759,0.852]","0.499 [0.431,0.566]","-0.811 [-0.867,-0.756]"
3,first_person_sentiment_positive,"27.724 [12.590,45.251]","29.957 [14.480,48.471]","0.878 [0.844,0.907]","0.863 [0.824,0.899]","0.821 [0.776,0.858]","2.202 [1.870,2.549]"
4,mattr_10,"0.936 [0.792,0.991]","0.948 [0.776,0.992]","0.854 [0.799,0.904]","0.812 [0.761,0.858]","0.729 [0.637,0.818]","0.012 [0.011,0.014]"
5,mattr_100,"0.805 [0.561,0.983]","0.855 [0.644,0.992]","0.977 [0.971,0.983]","0.976 [0.969,0.981]","0.788 [0.759,0.815]","0.046 [0.044,0.049]"
6,mattr_25,"0.864 [0.756,0.984]","0.897 [0.758,0.992]","0.934 [0.917,0.948]","0.924 [0.903,0.942]","0.707 [0.658,0.750]","0.030 [0.028,0.031]"
7,mattr_5,"0.977 [0.862,0.995]","0.980 [0.850,0.997]","0.776 [0.658,0.877]","0.685 [0.604,0.758]","0.750 [0.625,0.861]","0.003 [0.002,0.005]"
8,mattr_50,"0.824 [0.669,0.983]","0.866 [0.743,0.992]","0.965 [0.956,0.973]","0.961 [0.950,0.971]","0.752 [0.716,0.784]","0.040 [0.038,0.042]"
9,second_order_sentence_tangeniality,"0.439 [0.298,0.671]","0.435 [0.330,0.624]","0.820 [0.760,0.875]","0.773 [0.704,0.831]","0.806 [0.745,0.859]","-0.005 [-0.008,-0.002]"


In [6]:
L_TURNS = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    # discourse (per-turn)
    "first_order_sentence_tangeniality",
    "second_order_sentence_tangeniality",
    "turn_to_turn_tangeniality",
    "semantic_perplexity",
    "semantic_perplexity_5",
    "semantic_perplexity_11",
    "semantic_perplexity_15",
]

# 2) Turns files (turns_<pid>.csv), aggregated per participant by median:
stats_turns = check_statistic(
    features=L_TURNS,  # your per-turn feature list
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_gemma_new_json_split-1-3-26",#result_oppenwillis_eng_norm_gemma_updated_sentiment_FIXED_21-2-26",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#"/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_gemma_new_json_split-1-3-26",#result_oppenwillis_ukr_norm_gemma_updated_sentiment_FIXED_21-2-26",
    file_kind="turns",
    turns_agg="mean",
    # participant_only=True,
)

stats_turns.drop(columns=["note", 'n'], inplace=True)
stats_turns

/var/folders/zz/9j6cqjd91k7190vj479mr1jw0000gn/T/ipykernel_65048/2932081485.py:140: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.spearmanr(a, b, nan_policy="omit")


,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_person_percentage,"8.553 [2.967,15.135]","6.433 [1.648,12.951]","0.935 [0.919,0.949]","0.928 [0.906,0.946]","0.497 [0.445,0.550]","-2.233 [-2.336,-2.131]"
1,first_person_sentiment_negative,"2.603 [0.627,5.773]","1.607 [0.367,3.379]","0.859 [0.820,0.891]","0.848 [0.807,0.884]","0.384 [0.336,0.428]","-0.983 [-1.038,-0.926]"
2,first_person_sentiment_positive,"23.956 [15.371,37.489]","27.651 [18.549,42.312]","0.896 [0.870,0.919]","0.883 [0.850,0.912]","0.608 [0.560,0.656]","3.897 [3.667,4.120]"
3,mattr_10,"0.951 [0.909,0.989]","0.968 [0.929,0.997]","0.682 [0.608,0.746]","0.683 [0.605,0.750]","0.380 [0.319,0.438]","0.016 [0.014,0.017]"
4,mattr_100,"0.903 [0.734,0.989]","0.937 [0.839,0.996]","0.954 [0.942,0.965]","0.947 [0.931,0.960]","0.641 [0.603,0.673]","0.034 [0.032,0.037]"
5,mattr_25,"0.917 [0.848,0.989]","0.944 [0.889,0.996]","0.906 [0.883,0.927]","0.900 [0.871,0.925]","0.559 [0.513,0.602]","0.026 [0.025,0.028]"
6,mattr_5,"0.978 [0.953,0.994]","0.986 [0.950,0.999]","0.414 [0.288,0.530]","0.384 [0.273,0.491]","0.299 [0.200,0.394]","0.007 [0.006,0.008]"
7,mattr_50,"0.906 [0.783,0.989]","0.939 [0.854,0.996]","0.944 [0.929,0.956]","0.939 [0.921,0.954]","0.627 [0.586,0.662]","0.032 [0.029,0.034]"
8,semantic_perplexity,"32615.520 [118.060,15877003.996]","4347.707 [120.247,279237.989]","0.016 [-0.048,0.090]","0.169 [0.047,0.292]","0.000 [-0.003,0.003]","-387184.933 [-592151.572,-216832.445]"
9,semantic_perplexity_11,"34581.748 [1174.242,15877642.916]","11325.384 [4472.236,285440.184]","0.015 [-0.048,0.090]","0.098 [-0.018,0.213]","0.000 [-0.003,0.003]","-382818.968 [-587718.401,-212425.952]"
